In [1]:
# Menyiapkan dataset dengan tiga sumber  data

import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [10]:
# A. Extract

from pyspark.sql import SparkSession

# 1. Inisialisasi SparkSession
spark = SparkSession.builder.appName("Tugas6_ETL_Produksi").master("local[*]").getOrCreate()

# 2. Proses Extract
df_trx = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

# 3. Verifikasi jumlah baris dan skema
print("--- Data Transaksi ---")
print("Jumlah baris:", df_trx.count())
df_trx.printSchema()

print("--- Data Produk ---")
print("Jumlah baris:", df_produk.count())
df_produk.printSchema()

print("--- Data Ulasan ---")
print("Jumlah baris:", df_ulasan.count())
df_ulasan.printSchema()

--- Data Transaksi ---
Jumlah baris: 5000
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

--- Data Produk ---
Jumlah baris: 30
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

--- Data Ulasan ---
Jumlah baris: 3500
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [6]:
# B. Transform (Penggabungan)

from pyspark.sql.functions import col

# 1. Penggabungan Transaksi dan Produk (Inner Join)
df_etl = df_trx.join(df_produk, on="product_id", how="inner")

# 2. Penggabungan Hasil dengan Data Ulasan (Left Join)
df_etl = df_etl.join(df_ulasan, on="order_id", how="left")

# 3. Pengayaan Data: Menghitung total pendapatan
df_etl = df_etl.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

df_etl.show(5)

+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|order_id|product_id|unit_terjual|            tanggal| harga|    kategori|nama_produk|rating|total_pendapatan|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|     TX0|        21|           2|2026-10-10 00:00:00| 25000|  Elektronik|  Produk-21|     1|           50000|
|     TX1|         8|           6|2026-10-25 00:00:00|250000|     Fashion|   Produk-8|  NULL|         1500000|
|     TX2|        25|           4|2026-10-13 00:00:00| 25000|  Elektronik|  Produk-25|     2|          100000|
|     TX3|         3|           4|2026-10-18 00:00:00| 75000|     Fashion|   Produk-3|     5|          300000|
|     TX4|        19|           6|2026-10-07 00:00:00| 75000|Rumah Tangga|  Produk-19|  NULL|          450000|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
o

In [7]:
# C. Transform (Penanganan Data Kosong & Pengayaan)

# 1. Menambahkan kolom indikator keberadaan ulasan (sebelum mengisi nilai kosong)
df_etl = df_etl.withColumn("ada_ulasan", col("rating").isNotNull())

# 2. Menangani Missing Value pada kolom rating
df_etl = df_etl.fillna({"rating": 0})

df_etl.select("order_id", "product_id", "rating", "ada_ulasan", "total_pendapatan").show(5)

+--------+----------+------+----------+----------------+
|order_id|product_id|rating|ada_ulasan|total_pendapatan|
+--------+----------+------+----------+----------------+
|     TX0|        21|     1|      true|           50000|
|     TX1|         8|     0|     false|         1500000|
|     TX2|        25|     2|      true|          100000|
|     TX3|         3|     5|      true|          300000|
|     TX4|        19|     0|     false|          450000|
+--------+----------+------+----------+----------------+
only showing top 5 rows



In [12]:
# D. Load

# 1. Memastikan direktori target tersedia di HDFS
!hdfs dfs -mkdir -p /user/Grimmer/tugas6/hasil_etl

# 2. Menyimpan data dengan format Parquet dan dipartisi berdasarkan kategori
df_etl.write.mode("overwrite").partitionBy("kategori").parquet("hdfs://localhost:9000/user/Grimmer/tugas6/hasil_etl")

# 3. Verifikasi struktur folder
print("--- Struktur Direktori HDFS ---")
!hdfs dfs -ls -R /user/Grimmer/tugas6/hasil_etl

# 4. Membaca kembali dari HDFS untuk memastikan data tersimpan utuh
df_final = spark.read.parquet("hdfs://localhost:9000/user/Grimmer/tugas6/hasil_etl")
print("\nJumlah baris data hasil akhir di HDFS:", df_final.count())

[Stage 53:>                                                         (0 + 1) / 1]

--- Struktur Direktori HDFS ---


-rw-r--r--   3 Grimmer supergroup          0 2026-09-30 15:16 /user/Grimmer/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - Grimmer supergroup          0 2026-09-30 15:16 /user/Grimmer/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 Grimmer supergroup      16361 2026-09-30 15:16 /user/Grimmer/tugas6/hasil_etl/kategori=Elektronik/part-00000-7eec3fce-493d-485e-b30f-a7fa3c7f4eb3.c000.snappy.parquet
drwxr-xr-x   - Grimmer supergroup          0 2026-09-30 15:16 /user/Grimmer/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 Grimmer supergroup      11131 2026-09-30 15:16 /user/Grimmer/tugas6/hasil_etl/kategori=Fashion/part-00000-7eec3fce-493d-485e-b30f-a7fa3c7f4eb3.c000.snappy.parquet
drwxr-xr-x   - Grimmer supergroup          0 2026-09-30 15:16 /user/Grimmer/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 Grimmer supergroup      10389 2026-09-30 15:16 /user/Grimmer/tugas6/hasil_etl/kategori=Kesehatan/part-00000-7eec3fce-493d-485e-b30f-a7fa3c7f4eb3.c000.snappy.parquet
drwxr-xr-x   - Grimmer s

In [9]:
# E. Insight Akhir

from pyspark.sql.functions import avg, round

print("--- Persentase Transaksi dengan Ulasan per Kategori ---")
df_insight = df_final.groupBy("kategori") \
    .agg((round(avg(col("ada_ulasan").cast("int")) * 100, 2)).alias("persentase_ulasan")) \
    .orderBy("persentase_ulasan")

df_insight.show()

--- Persentase Transaksi dengan Ulasan per Kategori ---
+------------+-----------------+
|    kategori|persentase_ulasan|
+------------+-----------------+
|     Makanan|            68.84|
|   Kesehatan|            68.89|
|     Fashion|            70.14|
|Rumah Tangga|            70.55|
|  Elektronik|            70.66|
+------------+-----------------+



Kategori Makanan memiliki persentase transaksi dengan ulasan terendah, yaitu hanya 68,84%. Karena ulasan sangat penting untuk meyakinkan calon pembeli baru, tim marketing perlu memfokuskan strategi pada kategori ini.